In [ ]:
%uv add pymupdf sentence-transformers pymongo fastapi uvicorn python-dotenv google-genai

In [ ]:
import re
from pathlib import Path

import fitz


def clean_line(line: str) -> str:
    line = line.replace("u00ad", "")
    line = re.sub(r"s+", " ", line)
    return line.strip()

In [ ]:
def is_heading(line: str) -> bool:
    line = line.strip()

    if not line:
        return False

    if len(line) > 100:
        return False

    words = line.split()

    # obvious headings
    if line.isupper() and len(words) <= 12:
        return True

    if len(words) <= 8 and line.istitle():
        return True

    known_headings = {
        "Program Overview",
        "Program Vision",
        "Program Mission",
        "Faculties of Healthcare Program",
        "Syllabus Structure",
        "Career Opportunities",
        "Placement Facts",
        "Recruiters",
        "Alumni",
        "Eligibility",
    }

    return line in known_headings

In [ ]:
def extract_pages(pdf_path: str):
    doc = fitz.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc, start=1):
        raw_text = page.get_text("text")

        lines = [
            clean_line(x)
            for x in raw_text.splitlines()
        ]

        lines = [x for x in lines if x]

        pages.append({
            "page": page_num,
            "lines": lines
        })

    return pages

In [ ]:
def build_blocks(page_lines):
    """
    Convert lines into semantic blocks.
    Headings begin a new block.
    """

    blocks = []
    current_heading = "Unknown"
    current_lines = []

    def flush():
        nonlocal current_lines

        if current_lines:
            blocks.append({
                "section": current_heading,
                "text": " ".join(current_lines).strip()
            })

        current_lines = []

    for line in page_lines:

        if is_heading(line):
            flush()
            current_heading = line

        else:
            current_lines.append(line)

    flush()

    return blocks

In [ ]:
def chunk_blocks(
    blocks,
    max_words=320,
    overlap_words=50
):
    """
    Pack semantic blocks into reasonably sized chunks.
    """

    chunks = []
    current = []

    for block in blocks:

        candidate = current + [block["text"]]
        candidate_text = " ".join(candidate)

        if current and len(candidate_text.split()) > max_words:

            chunk_text = " ".join(current)

            chunks.append({
                "section": block["section"],
                "text": chunk_text
            })

            # overlap from the previous chunk
            words = chunk_text.split()
            overlap = " ".join(words[-overlap_words:])

            current = [overlap, block["text"]]

        else:
            current.append(block["text"])

    if current:
        chunks.append({
            "section": blocks[-1]["section"],
            "text": " ".join(current)
        })

    return chunks

In [ ]:
def process_pdf(pdf_path: str):

    pages = extract_pages(pdf_path)

    all_chunks = []

    for page_data in pages:

        blocks = build_blocks(
            page_data["lines"]
        )

        chunks = chunk_blocks(blocks)

        for idx, chunk in enumerate(chunks):

            all_chunks.append({
                "chunk_id": f"{Path(pdf_path).stem}_p{page_data['page']}_{idx}",
                "doc_id": Path(pdf_path).stem,
                "source": Path(pdf_path).name,
                "page": page_data["page"],
                "section": chunk["section"],
                "text": chunk["text"]
            })

    return all_chunks

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B"
)

documents = [chunk["text"] for chunk in chunks]

document_embeddings = model.encode(
    documents,
    batch_size=16,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

for chunk, embedding in zip(chunks, document_embeddings):
    chunk["embedding"] = embedding.tolist()

In [ ]:
query = "Who are the faculty of the Healthcare Program?"

query_embedding = model.encode(
    [query],
    prompt_name="query",
    normalize_embeddings=True
)[0]